# Análisis Global de Clima y Energía 2020–2024
## Paso 3: Dashboard de visualizaciones interactivas

**Autor:** Miguel Sierra  
**Objetivo:** Consolidar los hallazgos de los pasos anteriores en visualizaciones
interactivas ejecutivas, organizadas por dimensión de análisis.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("../data/dataset_limpio.csv", parse_dates=["date"])
df["year"]  = df["date"].dt.year
df["month"] = df["date"].dt.month

regiones = {
    "Germany": "Europa", "France": "Europa", "United Kingdom": "Europa",
    "Italy": "Europa", "Spain": "Europa", "Sweden": "Europa",
    "Norway": "Europa", "Netherlands": "Europa", "Poland": "Europa",
    "Turkey": "Europa",
    "United States": "América", "Canada": "América",
    "Mexico": "América", "Brazil": "América",
    "China": "Asia-Pacífico", "India": "Asia-Pacífico",
    "Japan": "Asia-Pacífico", "Indonesia": "Asia-Pacífico",
    "Australia": "Asia-Pacífico",
    "South Africa": "África"
}
df["region"] = df["country"].map(regiones)

resumen = pd.read_csv("../data/resumen_por_pais.csv")
resumen["region"] = resumen["country"].map(regiones)

print("✅ Datos cargados correctamente")

✅ Datos cargados correctamente


In [2]:
kpis = {
    "Temperatura promedio global (°C)": df["avg_temperature"].mean(),
    "CO₂ promedio global":              df["co2_emission"].mean(),
    "% Renovable promedio global":      df["renewable_share"].mean(),
    "Precio energía promedio":          df["energy_price"].mean(),
    "Consumo energético promedio":      df["energy_consumption"].mean(),
    "Índice industrial promedio":       df["industrial_activity_index"].mean(),
}

fig = go.Figure()

for i, (nombre, valor) in enumerate(kpis.items()):
    fig.add_trace(go.Indicator(
        mode="number",
        value=round(valor, 2),
        title={"text": nombre, "font": {"size": 13}},
        number={"font": {"size": 28}},
        domain={"row": i // 3, "column": i % 3}
    ))

fig.update_layout(
    title=dict(text="KPIs Globales — Promedio 2020–2024", font=dict(size=16)),
    grid={"rows": 2, "columns": 3, "pattern": "independent"},
    height=320,
    margin=dict(t=60, b=20, l=20, r=20)
)
fig.show()

In [3]:
temp_pais = df.groupby("country")["avg_temperature"].mean().reset_index()
temp_pais.columns = ["country", "temp_promedio"]

fig = px.choropleth(
    temp_pais,
    locations="country",
    locationmode="country names",
    color="temp_promedio",
    color_continuous_scale="RdYlBu_r",
    title="Temperatura promedio por país (2020–2024)",
    labels={"temp_promedio": "°C promedio"},
    height=450
)
fig.update_layout(
    coloraxis_colorbar=dict(title="°C"),
    geo=dict(showframe=False, showcoastlines=True)
)
fig.show()

In [4]:
temp_region_año = (
    df.groupby(["year","region"])["avg_temperature"]
    .mean().reset_index()
)

fig = px.area(
    temp_region_año, x="year", y="avg_temperature",
    color="region", line_group="region",
    title="Evolución de temperatura promedio anual por región",
    labels={"avg_temperature": "Temperatura promedio (°C)", "year": "Año"},
    height=420
)
fig.update_layout(xaxis=dict(tickmode="linear", dtick=1))
fig.show()

In [5]:
temp_heatmap = (
    df.groupby(["country","month"])["avg_temperature"]
    .mean().reset_index()
)
temp_pivot = temp_heatmap.pivot(index="country", columns="month", values="avg_temperature")
temp_pivot.columns = ["Ene","Feb","Mar","Abr","May","Jun",
                       "Jul","Ago","Sep","Oct","Nov","Dic"]

fig = px.imshow(
    temp_pivot,
    color_continuous_scale="RdYlBu_r",
    title="Temperatura promedio por país y mes",
    labels=dict(color="°C", x="Mes", y="País"),
    aspect="auto",
    height=580
)
fig.update_layout(coloraxis_colorbar=dict(title="°C"))
fig.show()

In [6]:
co2_pais = df.groupby("country")["co2_emission"].mean().reset_index()
co2_pais.columns = ["country","co2_promedio"]

fig = px.choropleth(
    co2_pais,
    locations="country",
    locationmode="country names",
    color="co2_promedio",
    color_continuous_scale="Reds",
    title="Emisiones promedio de CO₂ por país (2020–2024)",
    labels={"co2_promedio": "CO₂ promedio"},
    height=450
)
fig.update_layout(geo=dict(showframe=False, showcoastlines=True))
fig.show()

In [7]:
co2_año_pais = (
    df.groupby(["year","country","region"])["co2_emission"]
    .mean().reset_index()
)

fig = px.line(
    co2_año_pais, x="year", y="co2_emission",
    color="country", facet_col="region",
    facet_col_wrap=2, markers=True,
    title="Tendencia de CO₂ por país y región (2020–2024)",
    labels={"co2_emission": "CO₂ promedio", "year": "Año"},
    height=700
)
fig.update_layout(xaxis=dict(tickmode="linear", dtick=1), showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

In [8]:
bubble = (
    df.groupby(["country","region"])
    .agg(
        co2=("co2_emission","mean"),
        industrial=("industrial_activity_index","mean"),
        consumo=("energy_consumption","mean"),
        renovable=("renewable_share","mean")
    ).reset_index()
)

fig = px.scatter(
    bubble,
    x="industrial", y="co2",
    size="consumo", color="region",
    text="country",
    title="Actividad industrial vs CO₂ (tamaño = consumo energético)",
    labels={
        "industrial": "Índice de actividad industrial",
        "co2": "CO₂ promedio",
        "consumo": "Consumo energético"
    },
    height=550
)
fig.update_traces(textposition="top center", textfont_size=9)
fig.show()

In [9]:
renov_pais = df.groupby("country")["renewable_share"].mean().reset_index()
renov_pais.columns = ["country","renovable_promedio"]

fig = px.choropleth(
    renov_pais,
    locations="country",
    locationmode="country names",
    color="renovable_promedio",
    color_continuous_scale="Greens",
    title="Participación promedio de energías renovables por país (2020–2024)",
    labels={"renovable_promedio": "% Renovable"},
    height=450
)
fig.update_layout(geo=dict(showframe=False, showcoastlines=True))
fig.show()

In [10]:
media_global = df["renewable_share"].mean()

renov_rank = (
    df.groupby(["country","region"])["renewable_share"]
    .mean().reset_index()
    .sort_values("renewable_share", ascending=True)
)

fig = px.bar(
    renov_rank, x="renewable_share", y="country",
    color="region", orientation="h",
    title="Ranking de energías renovables vs promedio global",
    labels={"renewable_share": "% Renovable promedio", "country": "País"},
    height=620
)
fig.add_vline(
    x=media_global, line_dash="dash",
    line_color="black", line_width=1.5,
    annotation_text=f"Promedio global: {media_global:.1f}%",
    annotation_position="top right",
    annotation_font_size=11
)
fig.show()

In [11]:
fig = px.scatter(
    resumen,
    x="renovable_pct", y="precio_energia",
    size="co2_promedio", color="region",
    text="country",
    title="Renovables vs Precio energía vs CO₂ — vista multidimensional",
    labels={
        "renovable_pct": "% Renovable promedio",
        "precio_energia": "Precio de energía promedio",
        "co2_promedio": "CO₂ promedio"
    },
    height=540,
    size_max=40
)
fig.update_traces(textposition="top center", textfont_size=9)
fig.show()

In [14]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "CO₂ promedio por región",
        "% Renovable promedio por región",
        "Consumo energético por región",
        "Precio de energía por región"
    ]
)

metricas = ["co2_emission","renewable_share","energy_consumption","energy_price"]
colores  = ["#E24B4A","#1D9E75","#378ADD","#EF9F27"]
posiciones = [(1,1),(1,2),(2,1),(2,2)]

for metrica, color, (row, col) in zip(metricas, colores, posiciones):
    datos = (df.groupby("region")[metrica]
             .mean().reset_index()
             .sort_values(metrica, ascending=False))
    fig.add_trace(
        go.Bar(
            x=datos["region"],
            y=datos[metrica].round(2),
            marker_color=color,
            name=metrica,
            showlegend=False
        ),
        row=row, col=col
    )

fig.update_layout(
    title=dict(text="Comparativo de indicadores clave por región", font=dict(size=15)),
    height=600,
    template="plotly_white"
)
fig.show()

In [13]:
# Guardamos cada figura como HTML interactivo para incluir en la landing page
import plotly.io as pio

# Recreamos la figura del panel resumen para exportar
fig_export = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "CO₂ promedio por región",
        "% Renovable promedio por región",
        "Consumo energético por región",
        "Precio de energía por región"
    ]
)

for metrica, color, (row, col) in zip(metricas, colores, posiciones):
    datos = (df.groupby("region")[metrica]
             .mean().reset_index()
             .sort_values(metrica, ascending=False))
    fig_export.add_trace(
        go.Bar(x=datos["region"], y=datos[metrica].round(2),
               marker_color=color, showlegend=False),
        row=row, col=col
    )

fig_export.update_layout(
    title="Comparativo de indicadores clave por región",
    height=600, template="plotly_white"
)

pio.write_html(fig_export, file="../landing_page/dashboard_resumen.html",
               include_plotlyjs="cdn", full_html=True)

print("✅ Dashboard exportado en: landing_page/dashboard_resumen.html")
print("   Puedes abrir ese archivo en tu navegador directamente.")

✅ Dashboard exportado en: landing_page/dashboard_resumen.html
   Puedes abrir ese archivo en tu navegador directamente.


## Conclusiones — Paso 3

| Panel | Hallazgo visual clave |
|---|---|
| Temperatura | Asia-Pacífico domina en calor; Europa muestra los ciclos estacionales más marcados |
| CO₂ | Alta dispersión entre países dentro de la misma región |
| Renovables | Mayoría de países están por debajo del promedio global |
| Multi-dim | No hay correlación clara entre más renovables y menor precio de energía |
| Resumen | África lidera en menor consumo e industrial; Asia-Pacífico en mayor temperatura |

**Lo que queda para el Paso 4:**
Con las visualizaciones listas, el siguiente paso profundiza en
las correlaciones estadísticas formales, regresiones y los
insights finales que irán al documento teórico y la presentación.